## Executive Summary

**Purpose:** Every diameter, tooth thickness and inspection dimension of an involute spur gear, external or internal, with profile shift.

**What it does:** `GearGeometry` holds one gear (teeth, module, pressure angle, profile shift, basic rack, face width) and derives its reference, base, tip and root diameters, its tooth thickness, the span measurement over *k* teeth and the measurement over two balls or pins. `operating_pressure_angle` and `centre_distance` solve a meshing pair, and `contact_ratio` checks the overlap of the two tooth profiles.

**Units:** gear work is done in millimetres, newtons and MPa throughout `emachines.gears`, because that is what gear standards and gear drawings use. Angles are in radians inside the code and in degrees in reports.

**Sign convention (ISO 21771):** an internal gear is entered with a positive tooth count and `internal=True`. Internally the code gives it a negative tooth count, so one set of formulas covers both kinds of gear. A positive profile shift *x* always adds material to the tooth: an external tooth gets thicker at its base, an internal tooth gets thicker and its tip moves toward the centre.

## How It Fits Into the Gear Design Workflow

**Upstream:** nothing; this is the base of the `gears` module.

**Downstream:** `gears.layout` (tooth counts, centre distance and profile shift of a planetary set), `gears.rating` (ISO 6336 strength), `gears.manufacturing` (gear data sheets and tolerances).

In [ ]:
#| default_exp gears.involute

In [ ]:
#| export
from __future__ import annotations

import math
from dataclasses import dataclass, replace

import numpy as np

In [ ]:
#| hide
# drawing helpers for the explanatory figures (not exported)
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Arc, Polygon, Wedge
from emachines.gears import involute as _pkg              # the exported module, so figures can sit ahead of the definitions
from emachines.gears.manufacturing import gear_outline

INK, BLUE, RED, GREEN = "#222222", "#1f5fa8", "#b8322a", "#2a8a4a"
C_GEAR, C_TOOL = "#f6e7c8", "#dfe7f1"

def _rot(P, a):
    c, s_ = np.cos(a), np.sin(a)
    return np.asarray(P) @ np.array([[c, s_], [-s_, c]])

def _label(ax, text, xy, xytext, color=INK, ha="left", fs=9):
    ax.annotate(text, xy=xy, xytext=xytext, fontsize=fs, color=color, ha=ha, va="center",
                arrowprops=dict(arrowstyle="-", color=color, lw=0.7, shrinkA=0, shrinkB=0))

def _dim(ax, p0, p1, text="", off=(0, 0), color=BLUE, fs=9):
    ax.annotate("", xy=p1, xytext=p0, arrowprops=dict(arrowstyle="<->", color=color, lw=0.8, shrinkA=0, shrinkB=0))
    if text:
        ax.text((p0[0] + p1[0]) / 2 + off[0], (p0[1] + p1[1]) / 2 + off[1], text, color=color, fontsize=fs,
                ha="center", va="center", bbox=dict(fc="white", ec="none", pad=0.6, alpha=0.85))

def _arc(ax, r, a0, a1, color, ls="--", lw=0.9, c=(0, 0)):
    ax.add_patch(Arc(c, 2 * r, 2 * r, theta1=np.degrees(a0), theta2=np.degrees(a1), color=color, ls=ls, lw=lw))

def _rack_curve(m, alpha, ha, hf, rho, x0, x1, n_fil=12):
    "Outline of the basic rack (teeth up, tooth centred at x = 0, datum line y = 0) from x0 to x1."
    p, ta = np.pi * m, np.tan(alpha)
    cy = -hf * m + rho * m
    cx = rho * m / np.cos(alpha) + p / 4 - cy * ta              # fillet centre right of the tooth at 0
    t0 = np.pi + alpha                                         # tangent point on the flank
    one = [(-p / 4 + ha * m * ta, ha * m), (p / 4 - ha * m * ta, ha * m)]
    one += [(cx + rho * m * np.cos(t), cy + rho * m * np.sin(t)) for t in np.linspace(t0, 1.5 * np.pi, n_fil)]
    one += [(p - x_, y_) for x_, y_ in reversed(one[2:])]       # mirrored fillet of the next tooth
    one = np.array(one)
    pts = np.vstack([one + [k * p, 0] for k in range(int(np.floor(x0 / p)) - 1, int(np.ceil(x1 / p)) + 1)])
    return pts[(pts[:, 0] >= x0) & (pts[:, 0] <= x1)]

def _circ(ax, r, style="--", color=BLUE, lw=0.7, c=(0, 0)):
    ax.add_patch(Circle(c, r, fill=False, ls=style, ec=color, lw=lw))

## Notation

| Symbol | Meaning | Units |
|:--:|:--------------------------------|:--:|
| $z$ | number of teeth (negative for an internal gear inside the formulas) | – |
| $m$ | module | mm |
| $\alpha$ | reference (tool) pressure angle | rad |
| $x$ | profile shift coefficient | – |
| $h_{aP}^*$ | basic rack addendum, per unit module | – |
| $h_{fP}^*$ | basic rack dedendum, per unit module | – |
| $\rho_{fP}^*$ | basic rack root (tool tip) radius, per unit module | – |
| $d$ | reference diameter, $d = z\,m$ | mm |
| $d_b$ | base diameter, $d_b = d\cos\alpha$ | mm |
| $d_a$ | tip diameter | mm |
| $d_f$ | root diameter | mm |
| $s$ | tooth thickness on the reference circle (arc) | mm |
| $W_k$ | span measurement over $k$ teeth (base tangent length) | mm |
| $M_d$ | dimension over (external) or between (internal) two balls of diameter $D_M$ | mm |
| $\alpha_w$ | operating pressure angle of a meshing pair | rad |
| $a_w$ | operating centre distance | mm |
| $\varepsilon_\alpha$ | transverse contact ratio | – |

## The Involute Function

$$\operatorname{inv}\alpha = \tan\alpha - \alpha .$$

It is the polar angle of a point on the involute whose pressure angle is $\alpha$, so tooth thicknesses at different diameters are related through it. The inverse has no closed form. `inv_inverse` solves $\operatorname{inv}\alpha = v$ by Newton's method, using $\dfrac{d}{d\alpha}\operatorname{inv}\alpha = \tan^2\alpha$:

$$\alpha_{n+1} = \alpha_n - \frac{\tan\alpha_n - \alpha_n - v}{\tan^2\alpha_n},\qquad \alpha_0 = \min\!\left[(3v)^{1/3},\ 1.4\right],$$

where the start value comes from the small-angle series $\operatorname{inv}\alpha \approx \alpha^3/3$. It stops when $|\alpha_{n+1} - \alpha_n| < 10^{-14}$ rad (a few iterations).

In [ ]:
#| echo: false
def _inv_figure():
    fig, ax = plt.subplots(figsize=(10, 5.2))
    rb = 50.0
    ay = np.radians(42)                                  # pressure angle at the point Y
    ry = rb / np.cos(ay)
    th = np.linspace(0, 1.15, 200)                        # roll angle
    inv_pts = rb * np.c_[np.cos(th) + th * np.sin(th), np.sin(th) - th * np.cos(th)]
    ax.add_patch(Circle((0, 0), rb, fill=False, ec=GREEN, ls="--", lw=1.0))
    ax.plot(*inv_pts.T, color=INK, lw=1.8)
    # point Y, tangent point T, polar angle _pkg.inv(α_y)
    phi_T = np.tan(ay)                                   # roll angle to Y
    Y = rb * np.array([np.cos(phi_T) + phi_T * np.sin(phi_T), np.sin(phi_T) - phi_T * np.cos(phi_T)])
    T = rb * np.array([np.cos(phi_T), np.sin(phi_T)])
    S0 = np.array([rb, 0.0])
    th_Y = np.arctan2(Y[1], Y[0])
    for P_, c_ in ((S0, INK), (Y, RED), (T, GREEN)):
        ax.plot(*P_, "o", color=c_, ms=4)
    ax.plot([0, 1.3 * ry], [0, 0], color=INK, lw=0.6)
    ax.plot([0, Y[0] * 1.25], [0, Y[1] * 1.25], color=RED, lw=0.7)
    ax.plot([0, T[0]], [0, T[1]], color=GREEN, lw=0.7)
    ax.plot([T[0], Y[0]], [T[1], Y[1]], color=GREEN, lw=1.4)
    ax.add_patch(Circle((0, 0), ry, fill=False, ec=RED, ls="-.", lw=0.6))
    _arc(ax, 1.2 * ry, 0, th_Y, RED, ls="-", lw=1.3)
    _arc(ax, 20, th_Y, phi_T, GREEN, ls="-", lw=1.3)
    ax.text(1.22 * ry * np.cos(th_Y / 2) + 1, 1.22 * ry * np.sin(th_Y / 2), "inv $\\alpha_y$", color=RED, fontsize=11, va="center")
    am = (th_Y + phi_T) / 2
    ax.text(14 * np.cos(am) - 2, 14 * np.sin(am), "$\\alpha_y$", color=GREEN, fontsize=11, ha="center", va="center")
    # α_y also appears at Y between the radius OY and the tangent to the involute (normal is TY)
    ax.text(*(T + [-3, 4]), "T", color=GREEN, fontsize=11, ha="right")
    ax.text(*(Y + [3, -2]), "Y", color=RED, fontsize=11)
    ax.text(*(S0 + [1, -5]), "$S_0$", fontsize=11)
    ax.text(-3, -4, "O", fontsize=11, ha="right")
    _label(ax, "base circle, radius $r_b = d_b/2$", rb * np.array([np.cos(1.75), np.sin(1.75)]), (-20, 66), color=GREEN, ha="right")
    _label(ax, "involute: the path of the end of a string\nunwound from the base circle", inv_pts[185], (100, 8))
    _label(ax, "string TY: tangent to the base circle and\nnormal to the involute; length TY = arc $S_0$T", (T + Y) / 2, (100, 50), color=GREEN)
    _label(ax, "circle through Y, radius $r_y = r_b/\\cos\\alpha_y$", ry * np.array([np.cos(-0.45), np.sin(-0.45)]), (100, -35), color=RED)
    ax.set_xlim(-60, 175); ax.set_ylim(-42, 75); ax.set_aspect("equal"); ax.axis("off")
    ax.set_title("The involute: $\\alpha_y$ is the pressure angle at Y, inv $\\alpha_y = \\tan\\alpha_y - \\alpha_y$ is the polar angle of Y", fontsize=10)
    plt.tight_layout()
_inv_figure()

In [ ]:
#| export
def inv(alpha):
    "Involute function tan(α) − α (α in radians; works on arrays)."
    return np.tan(alpha) - alpha


def inv_inverse(value: float, tol: float = 1e-14) -> float:
    "The pressure angle α [rad] whose involute is `value` (0 ≤ value < ~1.5)."
    if value < 0:
        raise ValueError("involute is non-negative for 0 ≤ α < π/2")
    if value == 0:
        return 0.0
    a = (3.0 * value) ** (1.0 / 3.0)           # small-angle start: inv α ≈ α³/3
    a = min(a, 1.4)
    for _ in range(60):
        f = math.tan(a) - a - value
        da = f / math.tan(a) ** 2               # d(inv α)/dα = tan² α
        a -= da
        if abs(da) < tol:
            return a
    raise RuntimeError("inv_inverse did not converge")

In [ ]:
for deg in (14.5, 20.0, 25.0, 31.7):
    a = math.radians(deg)
    assert abs(inv_inverse(float(inv(a))) - a) < 1e-12
print(f"inv 20° = {inv(math.radians(20)):.6f}")   # 0.014904
assert abs(inv(math.radians(20)) - 0.0149044) < 1e-6

## Standard Modules

ISO 54 lists preferred modules (series I) and second-choice modules (series II). Gear cutting tools are stocked in these sizes, so a design should land on one of them.

In [ ]:
#| export
ISO54_SERIES_I = (0.1, 0.12, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0, 1.25, 1.5, 2.0, 2.5, 3.0,
                  4.0, 5.0, 6.0, 8.0, 10.0, 12.0, 16.0, 20.0, 25.0, 32.0, 40.0, 50.0)
ISO54_SERIES_II = (0.11, 0.14, 0.18, 0.22, 0.28, 0.35, 0.45, 0.55, 0.7, 0.9, 1.125, 1.375, 1.75, 2.25,
                   2.75, 3.5, 4.5, 5.5, 7.0, 9.0, 11.0, 14.0, 18.0, 22.0, 28.0, 36.0, 45.0)


def standard_modules(m_min: float = 0.0, m_max: float = 1e9, series_ii: bool = False) -> list[float]:
    "ISO 54 modules in [m_min, m_max], series I only unless `series_ii`."
    ms = ISO54_SERIES_I + (ISO54_SERIES_II if series_ii else ())
    return sorted(m for m in ms if m_min - 1e-12 <= m <= m_max + 1e-12)

In [ ]:
assert standard_modules(0.9, 2.1) == [1.0, 1.25, 1.5, 2.0]
assert 1.75 in standard_modules(1.6, 1.8, series_ii=True)

## `GearGeometry`: Theory

A spur gear is fully defined by $z$, $m$, $\alpha$, $x$, the basic rack of the cutting tool, and the face width $b$ (ISO 21771).

**Basic rack.** Every gear here is cut by a tool whose profile is the ISO 53 basic rack: straight flanks at the pressure angle $\alpha$, equal tooth and space on the datum line (each $p/2$, with pitch $p = \pi m$), addendum $h_{aP}^* m$, dedendum $h_{fP}^* m$ and root radius $\rho_{fP}^* m$. The module $m$ scales the whole profile.

In [ ]:
#| echo: false
def _rack_figure():
    fig, ax = plt.subplots(figsize=(11, 4.6))
    m_, al = 10.0, np.radians(20)
    p_ = np.pi * m_
    cur = _rack_curve(m_, al, 1.0, 1.25, 0.38, -1.2 * p_, 1.2 * p_)
    body = np.vstack([cur, [[cur[-1, 0], -2.2 * m_], [cur[0, 0], -2.2 * m_]]])
    ax.add_patch(Polygon(body, closed=True, fc=C_TOOL, ec=INK, lw=1.0))
    for y_, c_, ls_ in ((0, RED, "-."), (m_, BLUE, "--"), (-1.25 * m_, BLUE, "--"), (-m_, INK, ":")):
        ax.plot([-1.25 * p_, 1.25 * p_], [y_, y_], color=c_, ls=ls_, lw=0.8)
    ta = np.tan(al)
    # pitch, thickness and space on the datum line
    _dim(ax, (-p_ / 4, 1.75 * m_), (3 * p_ / 4, 1.75 * m_), "pitch $p = \\pi m$", color=INK)
    ax.plot([-p_ / 4, -p_ / 4], [0, 1.9 * m_], color=INK, lw=0.4); ax.plot([3 * p_ / 4, 3 * p_ / 4], [0, 1.9 * m_], color=INK, lw=0.4)
    _dim(ax, (-p_ / 4, 0), (p_ / 4, 0), "", color=RED); ax.text(0, 0.35 * m_, "$s_P = p/2$", color=RED, ha="center", fontsize=9)
    _dim(ax, (p_ / 4, 0), (3 * p_ / 4, 0), "", color=RED); ax.text(p_ / 2, 0.35 * m_, "$e_P = p/2$", color=RED, ha="center", fontsize=9)
    # heights on the left of the tooth at -p
    xh = -1.12 * p_
    _dim(ax, (xh, 0), (xh, m_)); ax.text(xh - 2, 0.5 * m_, "addendum $h_{aP} = 1.0\\,m$", color=BLUE, ha="right", va="center", fontsize=9)
    _dim(ax, (xh, -1.25 * m_), (xh, 0)); ax.text(xh - 2, -0.62 * m_, "dedendum $h_{fP} = 1.25\\,m$", color=BLUE, ha="right", va="center", fontsize=9)
    _dim(ax, (xh + 6, -1.25 * m_), (xh + 6, -m_), color=INK); ax.text(xh - 2, -1.6 * m_, "bottom clearance $c_P = 0.25\\,m$", color=INK, ha="right", va="center", fontsize=9)
    # pressure angle on the right flank of the tooth at 0
    F = np.array([p_ / 4, 0.0])
    ax.plot([F[0], F[0]], [0, 1.5 * m_], color=INK, lw=0.6, ls="--")
    ax.add_patch(Arc(F, 2.2 * m_, 2.2 * m_, theta1=90, theta2=110, color=GREEN, lw=1.3))
    ax.text(F[0] - 0.3 * m_, F[1] + 1.35 * m_, "$\\alpha$", color=GREEN, fontsize=12, ha="center")
    # root fillet radius in the space at +p/2
    cy = -1.25 * m_ + 0.38 * m_
    cx = 0.38 * m_ / np.cos(al) + p_ / 4 - cy * ta
    ax.plot([cx, cx + 0.38 * m_ * np.cos(np.radians(235))], [cy, cy + 0.38 * m_ * np.sin(np.radians(235))], color=GREEN, lw=0.8)
    ax.plot(cx, cy, "+", color=GREEN, ms=6)
    _label(ax, "root radius $\\rho_{fP} = 0.38\\,m$", (cx, cy), (1.32 * p_, -1.45 * m_), color=GREEN)
    _label(ax, "datum (reference) line: $s_P = e_P$ here", (1.2 * p_, 0), (1.32 * p_, 0), color=RED)
    _label(ax, "tip line", (1.2 * p_, m_), (1.32 * p_, m_), color=BLUE)
    _label(ax, "root line", (1.2 * p_, -1.25 * m_), (1.32 * p_, -0.75 * m_), color=BLUE)
    _label(ax, "pressure angle $\\alpha$ = 20°:\nflank against the normal to the datum line", F + [-0.1 * m_, 1.3 * m_], (1.32 * p_, 1.6 * m_), color=GREEN)
    ax.set_xlim(-2.5 * p_, 2.3 * p_); ax.set_ylim(-2.3 * m_, 2.4 * m_); ax.set_aspect("equal"); ax.axis("off")
    ax.set_title("ISO 53 basic rack, profile A (the reference tooth form every spur gear in this module is cut from)", fontsize=10)
    plt.tight_layout()
_rack_figure()

**Tooth count, module, pressure angle and profile shift.** The gear is generated by rolling its reference circle ($d = z\,m$) on the pitch line of a rack-shaped cutter. With $x = 0$ the cutter datum line is that pitch line; with profile shift the cutter is moved out by $x\,m$, which thickens the teeth and raises both the tip and root circles while $d$ and $d_b$ stay the same. Negative $x$ moves the cutter in. Small external gears need positive shift to avoid undercut (left panel: 14 teeth, x = 0, the cutter tip has eaten into the bottom of the flank).

In [ ]:
#| echo: false
def _gen_figure():
    fig, axs = plt.subplots(1, 2, figsize=(14, 4.4))
    for ax, xs in zip(axs, (0.0, 0.5)):
        m_, z_ = 10.0, 14
        gg = _pkg.GearGeometry(z=z_, m=m_, x=xs)
        r_ = gg.d / 2
        P = _rot(gear_outline(gg, n_flank=60, n_arc=20), np.pi / 2 + np.pi / z_)
        body = Polygon(P, closed=True, fc=C_GEAR, ec=INK, lw=0.9)
        ax.add_patch(body)
        body.set_clip_path(Wedge((0, 0), gg.da / 2 + 2, 58, 122, width=gg.da / 2 + 2 - (gg.df / 2 - 8), transform=ax.transData))
        # cutter = complement of the basic rack; its datum line sits x·m above the tangent to the reference circle
        p_ = np.pi * m_
        cur = _rack_curve(m_, gg.alpha, 1.0, 1.25, 0.38, -1.5 * p_, 1.5 * p_)
        cut = np.c_[cur[:, 0] - p_ / 2, cur[:, 1]]          # cutter tooth (a rack space) centred on x = 0
        cut = cut[(cut[:, 0] >= -1.35 * p_) & (cut[:, 0] <= 1.35 * p_)]
        yd = r_ + xs * m_
        tool = np.vstack([cut + [0, yd], [[cut[-1, 0], yd + 2.4 * m_], [cut[0, 0], yd + 2.4 * m_]]])
        ax.add_patch(Polygon(tool, closed=True, fc=C_TOOL, ec=INK, lw=0.9, alpha=0.9))
        ax.add_patch(Arc((0, 0), 2 * r_, 2 * r_, theta1=50, theta2=130, color=RED, ls="-.", lw=1.0))
        ax.plot([-1.4 * p_, 1.4 * p_], [r_, r_], color=RED, lw=0.6, ls=":")
        ax.plot([-1.4 * p_, 1.4 * p_], [yd, yd], color=RED, lw=1.0, ls="--")
        for rr, c_ in ((gg.da / 2, BLUE), (gg.df / 2, BLUE)):
            ax.add_patch(Arc((0, 0), 2 * rr, 2 * rr, theta1=55, theta2=125, color=c_, ls="--", lw=0.7))
        ax.plot(0, r_, "o", color=RED, ms=3.5)
        if xs:
            xd = 1.05 * p_
            _dim(ax, (xd, r_), (xd, yd), "", color=RED)
            ax.text(xd + 2, (r_ + yd) / 2, f"$x\\,m$ = {xs * m_:g} mm", color=RED, fontsize=9, va="center")
        xl = -1.55 * p_
        tips = [("cutter datum line", (-1.3 * p_, yd), yd + 8, RED),
                ("pitch line: tangent to the reference\ncircle, rolls on it without slip", (-1.3 * p_, r_), r_ - 2, RED),
                ("tip circle $d_a$", gg.da / 2 * np.array([np.cos(np.radians(124)), np.sin(np.radians(124))]), r_ - 16, BLUE),
                ("reference circle $d = z\\,m$", r_ * np.array([np.cos(np.radians(129)), np.sin(np.radians(129))]), r_ - 26, RED),
                ("root circle $d_f$", gg.df / 2 * np.array([np.cos(np.radians(124)), np.sin(np.radians(124))]), r_ - 36, BLUE)]
        for txt, xy, ty, c_ in tips:
            _label(ax, txt, xy, (xl, ty), color=c_, ha="right", fs=8.5)
        ax.text(0, gg.df / 2 - 13, f"x = {xs:+.1f}:  $d_a$ = {gg.da:.0f},  $d_f$ = {gg.df:.0f},  s = {gg.s:.1f} mm"
                + ("   (undercut)" if gg.undercut else ""), ha="center", fontsize=9)
        ax.set_title(f"z = {z_}, m = {m_:g} mm, α = 20°, x = {xs:+.1f}", fontsize=10)
        ax.set_xlim(-2.75 * p_, 1.9 * p_); ax.set_ylim(gg.df / 2 - 16, yd + 2.6 * m_); ax.set_aspect("equal"); ax.axis("off")
    fig.suptitle("Profile shift: the gear is cut by a rack-shaped cutter (blue) whose datum line is moved out by $x\\,m$. "
                 "z, m and α fix d and $d_b$; x changes tooth thickness, tip and root.", fontsize=10)
    plt.tight_layout()
_gen_figure()

**Diameters and thicknesses.** The resulting circles and tooth dimensions on one gear:

In [ ]:
#| echo: false
def _tooth_figure():
    ge = _pkg.GearGeometry(z=20, m=10.0, x=0.0, b=60, name="external")
    up = np.pi / 2
    pitch = 2 * np.pi / ge.z
    fig, ax = plt.subplots(figsize=(13, 4.4))
    P = _rot(gear_outline(ge, n_flank=60, n_arc=20), up)
    r, ra, rf, rb = ge.d / 2, ge.da / 2, ge.df / 2, ge.db / 2
    th0, th1 = 62, 118
    body = Polygon(P, closed=True, fc=C_GEAR, ec=INK, lw=1.0)
    ax.add_patch(body)
    body.set_clip_path(Wedge((0, 0), ra + 4, th0 - 6, th1 + 6, width=ra + 4 - (rf - 9), transform=ax.transData))
    for rr, col, ls in ((ra, BLUE, "--"), (r, RED, "-."), (rb, GREEN, "--"), (rf, BLUE, "--")):
        ax.add_patch(Arc((0, 0), 2 * rr, 2 * rr, theta1=th0, theta2=th1, ls=ls, color=col, lw=0.9))
    # circle names at the right-hand end of each arc
    for rr, col, nm, dy in ((ra, BLUE, "tip circle, diameter $d_a$", 1.5), (r, RED, "reference circle, diameter $d = z\\,m$", 0),
                            (rb, GREEN, "base circle, diameter $d_b = d\\cos\\alpha$", -1.0), (rf, BLUE, "root circle, diameter $d_f$", -2.5)):
        a = np.radians(th0)
        ax.text(rr * np.cos(a) + 2, rr * np.sin(a) + dy, nm, color=col, fontsize=9, ha="left", va="center")
    ax.plot([0, 0], [rf - 6, ra + 14], color=RED, lw=0.5, ls="-.")
    psi = ge.s / ge.d
    def arcdim(a0, a1, rr, text, col, toff):
        ax.add_patch(Arc((0, 0), 2 * rr, 2 * rr, theta1=np.degrees(a0), theta2=np.degrees(a1), color=col, lw=1.6))
        for a in (a0, a1):
            ax.plot([(rr - 2.5) * np.cos(a), (rr + 2.5) * np.cos(a)], [(rr - 2.5) * np.sin(a), (rr + 2.5) * np.sin(a)], color=col, lw=0.9)
        am = (a0 + a1) / 2
        ax.text((rr + toff) * np.cos(am), (rr + toff) * np.sin(am), text, color=col, fontsize=9, ha="center", va="center",
                bbox=dict(fc="white", ec="none", pad=0.5, alpha=0.9))
    arcdim(up - psi, up + psi, r, "tooth\nthickness $s$", RED, 4.5)
    arcdim(up + psi, up + pitch - psi, r, "space\nwidth $e$", BLUE, -4.5)
    arcdim(up - psi - pitch, up - psi, ra + 9, "pitch $p = \\pi m$", INK, 3.5)
    # radial dimensions in the space to the left of the centre tooth's neighbour
    a_d = up + 1.5 * pitch
    u = np.array([np.cos(a_d), np.sin(a_d)])
    _dim(ax, r * u, ra * u, "", color=BLUE)
    _dim(ax, rf * u, r * u, "", color=BLUE)
    a_h = np.radians(th1 + 3.5)
    uh = np.array([np.cos(a_h), np.sin(a_h)])
    _dim(ax, rf * uh, ra * uh, "", color=BLUE)
    # pressure angle on the right flank of the centre tooth, at the reference circle
    th = up - psi
    Pp = r * np.array([np.cos(th), np.sin(th)])
    T = rb * np.array([np.cos(th + ge.alpha), np.sin(th + ge.alpha)])
    v = Pp - T
    ax.plot([T[0], Pp[0] + 0.9 * v[0]], [T[1], Pp[1] + 0.9 * v[1]], color=GREEN, lw=1.1)
    ax.plot(*T, "o", color=GREEN, ms=3)
    tdir = np.array([np.sin(th), -np.cos(th)])
    ax.plot([Pp[0] - 10 * tdir[0], Pp[0] + 26 * tdir[0]], [Pp[1] - 10 * tdir[1], Pp[1] + 26 * tdir[1]], color=INK, lw=0.6, ls="--")
    a_t, a_n = np.degrees(np.arctan2(tdir[1], tdir[0])), np.degrees(np.arctan2(v[1], v[0]))
    ax.add_patch(Arc(Pp, 22, 22, theta1=min(a_t, a_n), theta2=max(a_t, a_n), color=GREEN, lw=1.3))
    am = np.radians((a_t + a_n) / 2)
    ax.text(Pp[0] + 15 * np.cos(am), Pp[1] + 15 * np.sin(am), "$\\alpha$", color=GREEN, fontsize=13, ha="center", va="center")
    # leaders on the right, top to bottom
    ax_r = 62
    _label(ax, "line of action: normal to the flank,\ntangent to the base circle (at T)", Pp + 0.75 * v, (ax_r, 122), color=GREEN)
    _label(ax, "tangent to the reference circle", Pp + 22 * tdir, (ax_r, 113), color=INK)
    _label(ax, "pressure angle $\\alpha$ = 20° between them", Pp + 9 * np.array([np.cos(am), np.sin(am)]), (ax_r, 106), color=GREEN)
    # leaders on the left, top to bottom
    rr_f = ra - 4
    hf_ = psi + float(_pkg.inv(ge.alpha)) - float(_pkg.inv(np.arccos(rb / rr_f)))
    flank_pt = rr_f * np.array([np.cos(up + hf_), np.sin(up + hf_)])
    fil = (rf + 1.2) * np.array([np.cos(up + pitch / 2 - 0.035), np.sin(up + pitch / 2 - 0.035)])
    ax_l = -66
    _label(ax, "tooth tip", (0, ra), (ax_l, 124), ha="right")
    _label(ax, "flank (involute)", flank_pt, (ax_l, 116), ha="right")
    _label(ax, "addendum $h_a = m$", (r + ra) / 2 * u, (ax_l, 106), ha="right", color=BLUE)
    _label(ax, "dedendum $h_f = 1.25\\,m$", (r + rf) / 2 * u, (ax_l, 97), ha="right", color=BLUE)
    _label(ax, "tooth depth $h = h_a + h_f = 2.25\\,m$", (ra + rf) / 2 * uh, (ax_l, 88), ha="right", color=BLUE)
    _label(ax, "root fillet (left by the hob tip radius)", fil, (ax_l, 79), ha="right")
    ax.text(0.5, -0.01, f"External spur gear: z = {ge.z}, m = {ge.m:g} mm (drawn large), α = 20°, x = 0. "
            "Face width b is the tooth length into the page.", ha="center", va="top", fontsize=9, transform=ax.transAxes)
    ax.set_xlim(-150, 165); ax.set_ylim(rf - 17, ra + 18)
    ax.set_aspect("equal"); ax.axis("off")
    plt.tight_layout()
_tooth_figure()

With the signed tooth count ($z<0$ for an internal gear, so $d<0$ too):

$$d = z m,\qquad d_b = d\cos\alpha,\qquad p = \pi m,\qquad p_b = p\cos\alpha$$

$$d_a = d + 2m\,(h_{aP}^* + x - k^*),\qquad d_f = d - 2m\,(h_{fP}^* - x)$$

where $k^*$ is an optional tip reduction (for an internal gear it is used to keep the ring tips clear of the planet roots and to lift them off the base circle). For an internal gear the formulas give negative diameters; their sizes are the diameters you would measure.

Arc tooth thickness on the reference circle, and on any other circle of diameter $d_y$ with pressure angle $\alpha_y = \arccos(d_b/d_y)$:

$$s = m\left(\frac{\pi}{2} + 2x\tan\alpha\right),\qquad s_y = d_y\left(\frac{s}{d} + \operatorname{inv}\alpha - \operatorname{inv}\alpha_y\right).$$

The tip thickness $s_a$ (tooth thickness on the tip circle) must stay above about $0.25\,m$ for case-hardened teeth, otherwise the tip through-hardens and chips.

**Undercut.** A rack-generated external gear is undercut (the cutter removes the bottom of the working involute) when

$$z < z_{min} = \frac{2\,(h_{aP0}^* - x - \rho_{aP0}^*(1-\sin\alpha))}{\sin^2\alpha},$$

with $h_{aP0}^* = h_{fP}^*$ the cutter addendum and $\rho_{aP0}^* = \rho_{fP}^*$ its tip radius. For the ISO 53 profile A rack ($\alpha = 20°$, $h_{fP}^* = 1.25$, $\rho_{fP}^* = 0.38$) this is 17 teeth at $x = 0$.

**In positive sizes**, as `GearGeometry` returns them (the internal-gear lines are the signed formulas above with $z<0$):

| | External gear | Internal gear |
|---|---|---|
| tip diameter | $d_a = d + 2m\,(h_{aP}^* + x - k^*)$ | $d_a = d - 2m\,(h_{aP}^* + x - k^*)$ |
| root diameter | $d_f = d - 2m\,(h_{fP}^* - x)$ | $d_f = d + 2m\,(h_{fP}^* - x)$ |
| tooth depth | $h = \lvert d_f - d_a\rvert/2 = m\,(h_{aP}^* + h_{fP}^* - k^*)$ | same |
| thickness at $d_y$ | $s_y = d_y\left(\dfrac{s}{d} + \operatorname{inv}\alpha - \operatorname{inv}\alpha_y\right)$ | $s_y = d_y\left(\dfrac{s}{d} - \operatorname{inv}\alpha + \operatorname{inv}\alpha_y\right)$ |
| tip thickness | $s_a = s_y$ at $d_y = \max(d_a, d_b)$ | same |

The internal tooth gets wider towards its root (larger diameter), which is why the sign of the involute terms flips.

**Root form diameter $d_{Ff}$** (where the generated involute ends and the fillet begins). The straight part of the cutter flank reaches

$$h_{lin} = m\left[h_{fP}^* - \rho_{fP}^*(1 - \sin\alpha) - x\right]$$

below the pitch line. The last involute point it generates lies on the line of action at a distance $g$ from the base-circle tangent point,

$$g = \frac{d\sin\alpha}{2} - \frac{h_{lin}}{\sin\alpha}\ \text{(external)},\qquad g = \frac{d\sin\alpha}{2} + \frac{h_{lin}}{\sin\alpha}\ \text{(internal)},\qquad d_{Ff} = 2\sqrt{r_b^2 + g^2}.$$

If $g < 0$ on an external gear the involute would have to start inside the base circle: the tooth is undercut and $d_{Ff} = d_b$. The internal value assumes a rack-type cutter; a pinion-type shaper gives a slightly different fillet, so confirm it with the cutter data.

In [ ]:
#| export
@dataclass(frozen=True)
class GearGeometry:
    """
    One involute spur gear (ISO 21771). Lengths in mm, angles in radians.

    Attributes
    ----------
    z        : int    Number of teeth, always entered positive
    m        : float  Module [mm]
    x        : float  Profile shift coefficient (positive adds material to the tooth, ISO 21771)
    b        : float  Face width [mm]
    alpha    : float  Reference pressure angle [rad] (default 20°)
    internal : bool   True for an internal (ring) gear
    ha       : float  Basic rack addendum coefficient h_aP* (default 1.0)
    hf       : float  Basic rack dedendum coefficient h_fP* (default 1.25)
    rho_f    : float  Basic rack root radius coefficient ρ_fP* (default 0.38, ISO 53 profile A)
    k_tip    : float  Tip reduction coefficient k* (default 0)
    name     : str    Label used in reports
    """
    z: int
    m: float
    x: float = 0.0
    b: float = 10.0
    alpha: float = math.radians(20.0)
    internal: bool = False
    ha: float = 1.0
    hf: float = 1.25
    rho_f: float = 0.38
    k_tip: float = 0.0
    name: str = "gear"

    def __post_init__(self):
        if self.z < 3:
            raise ValueError("z must be a positive tooth count ≥ 3 (use internal=True for a ring gear)")
        if self.m <= 0 or self.b <= 0:
            raise ValueError("module and face width must be positive")

    # --- signed quantities (ISO 21771: negative for internal gears) ---------
    @property
    def zs(self) -> int:
        "Signed tooth count (negative for an internal gear)."
        return -self.z if self.internal else self.z

    @property
    def d_signed(self) -> float: return self.zs * self.m

    # --- diameters (always positive) -----------------------------------------
    @property
    def d(self) -> float:
        "Reference diameter [mm]."
        return abs(self.d_signed)

    @property
    def db(self) -> float:
        "Base diameter [mm]."
        return self.d * math.cos(self.alpha)

    @property
    def da(self) -> float:
        "Tip diameter [mm] (for an internal gear: the tip circle inside the ring)."
        add = 2 * self.m * (self.ha + self.x - self.k_tip)
        return self.d - add if self.internal else self.d + add

    @property
    def df(self) -> float:
        "Root diameter [mm] (for an internal gear: the root circle at the outside of the teeth)."
        ded = 2 * self.m * (self.hf - self.x)
        return self.d + ded if self.internal else self.d - ded

    @property
    def p(self) -> float: return math.pi * self.m

    @property
    def pb(self) -> float: return self.p * math.cos(self.alpha)

    @property
    def h(self) -> float:
        "Tooth depth (tip to root) [mm]."
        return abs(self.df - self.da) / 2

    # --- tooth thickness -------------------------------------------------------
    @property
    def s(self) -> float:
        "Arc tooth thickness on the reference circle [mm], no allowance."
        return self.m * (math.pi / 2 + 2 * self.x * math.tan(self.alpha))

    def s_at(self, dy: float, s: float | None = None) -> float:
        "Arc tooth thickness on diameter `dy` [mm], given reference thickness `s` (default: nominal)."
        s = self.s if s is None else s
        if dy < self.db - 1e-9:
            raise ValueError("diameter is inside the base circle: no involute there")
        ay = math.acos(min(1.0, self.db / dy))
        if self.internal:   # the internal tooth widens toward its root (larger diameter)
            return dy * (s / self.d - inv(self.alpha) + inv(ay))
        return dy * (s / self.d + inv(self.alpha) - inv(ay))

    @property
    def sa(self) -> float:
        "Tooth thickness on the tip circle [mm]."
        return self.s_at(max(self.da, self.db))

    @property
    def z_min_undercut(self) -> float:
        "Smallest tooth count without undercut for this profile shift (rack-generated external gear)."
        a = self.alpha
        return 2 * (self.hf - self.x - self.rho_f * (1 - math.sin(a))) / math.sin(a) ** 2

    @property
    def undercut(self) -> bool:
        "True if a rack-generated external gear with this shift is undercut."
        return (not self.internal) and self.z < self.z_min_undercut - 1e-9

    @property
    def d_form(self) -> float:
        """
        Root form diameter d_Ff [mm]: where the generated involute meets the fillet.

        Exact for a rack-generated (hobbed) external gear. For an internal gear the
        mirrored rack formula is used; a pinion-type shaper cutter gives a slightly
        different fillet, so treat it as an estimate and confirm it with the cutter data.
        """
        a = self.alpha
        h_lin = self.m * (self.hf - self.rho_f * (1 - math.sin(a)) - self.x)   # straight-flank depth below pitch line
        if self.internal:
            g = self.d * math.sin(a) / 2 + h_lin / math.sin(a)
        else:
            g = self.d * math.sin(a) / 2 - h_lin / math.sin(a)
        if g < 0:   # generated involute stops inside the base circle: undercut
            return self.db
        return 2 * math.hypot(self.db / 2, g)

    def with_(self, **kw) -> "GearGeometry":
        "A copy with some fields changed."
        return replace(self, **kw)

### Example

A 17-tooth, module 1.5 sun with a +0.3 shift and the 67-tooth ring of the same planetary set.

In [ ]:
sun = GearGeometry(z=17, m=1.5, x=0.3, b=20, name="sun")
ring = GearGeometry(z=67, m=1.5, x=0.0, b=20, internal=True, name="ring")
for g in (sun, ring):
    print(f"{g.name:5s} d = {g.d:7.3f}  db = {g.db:7.3f}  da = {g.da:7.3f}  df = {g.df:7.3f}  "
          f"s = {g.s:.4f}  sa = {g.sa:.3f}  d_Ff = {g.d_form:.3f} mm")
assert math.isclose(sun.da, 25.5 + 2 * 1.5 * 1.3) and math.isclose(sun.df, 25.5 - 2 * 1.5 * 0.95)
assert math.isclose(ring.da, 100.5 - 3.0) and math.isclose(ring.df, 100.5 + 3.75)
assert not sun.undercut and GearGeometry(z=12, m=1).undercut
assert abs(GearGeometry(z=17, m=1).z_min_undercut - 17.1) < 0.2     # ISO profile A, x = 0

## Inspection Dimensions: Theory

Tooth thickness cannot be measured directly, so gear drawings specify it through a dimension the gear shop can measure: the span over $k$ teeth or the dimension over two balls (or pins). Both are linked exactly to the reference tooth thickness $s$.

In [ ]:
#| echo: false
def _meas_figure():
    fig, axs = plt.subplots(1, 2, figsize=(13, 6.2))
    gm = _pkg.GearGeometry(z=20, m=10.0, x=0.0)
    rb, ra, r_ = gm.db / 2, gm.da / 2, gm.d / 2
    # (a) span over k teeth: both anvils touch flanks on one base tangent (here the horizontal line y = r_b)
    ax = axs[0]
    k = _pkg.span_teeth(gm); W = _pkg.span_measurement(gm)
    up = np.pi / 2 + (0 if k % 2 else np.pi / gm.z)     # odd k: a tooth on top; even k: a space on top
    ax.add_patch(Polygon(_rot(gear_outline(gm, n_flank=60, n_arc=20), up), closed=True, fc=C_GEAR, ec=INK, lw=0.9))
    ax.add_patch(Circle((0, 0), rb, fill=False, ec=GREEN, ls="--", lw=0.9))
    ax.plot([-W / 2 - 15, W / 2 + 15], [rb, rb], color=GREEN, lw=1.1)
    for sgn in (-1, 1):
        x_ = sgn * W / 2
        ax.add_patch(Polygon([[x_, rb - 6], [x_, ra + 22], [x_ + sgn * 6, ra + 22], [x_ + sgn * 6, rb - 6]],
                             closed=True, fc="#c8c8c8", ec=INK, lw=0.8, zorder=3))
        ax.plot(x_, rb, "o", color=RED, ms=4, zorder=4)
    _dim(ax, (-W / 2, ra + 30), (W / 2, ra + 30), f"$W_k$ = {W:.3f} mm  (k = {k})", color=INK)
    for sgn in (-1, 1):
        ax.plot([sgn * W / 2, sgn * W / 2], [ra + 22, ra + 33], color=INK, lw=0.5)
    _label(ax, "disc micrometer anvils (parallel)", (W / 2 + 6, ra + 10), (W / 2 + 30, ra + 18))
    _label(ax, "contact points lie on one base tangent,\nso $W_k = (k-1)\\,p_b + s_b$", (W / 2, rb), (W / 2 + 30, rb - 8), color=RED)
    _label(ax, "base circle $d_b$", rb * np.array([np.cos(2.6), np.sin(2.6)]), (-W / 2 - 25, rb - 25), color=GREEN, ha="right")
    ax.set_title(f"Span measurement over k = {k} teeth (z = {gm.z}, m = {gm.m:g} mm)", fontsize=10)
    ax.set_xlim(-W / 2 - 95, W / 2 + 140); ax.set_ylim(rb - 32, ra + 42); ax.set_aspect("equal"); ax.axis("off")
    # (b) dimension over two balls (z even: balls in opposite spaces)
    ax = axs[1]
    D = _pkg.ball_diameter(gm); M, dc = _pkg.ball_measurement(gm)
    ax.add_patch(Polygon(_rot(gear_outline(gm, n_flank=40, n_arc=12), np.pi / gm.z), closed=True, fc=C_GEAR, ec=INK, lw=0.8))
    rM = (M - D) / 2
    for sgn in (-1, 1):
        ax.add_patch(Circle((sgn * rM, 0), D / 2, fc="#9a9a9a", ec=INK, lw=0.8))
        ax.add_patch(Polygon([[sgn * M / 2, -25], [sgn * M / 2, 25], [sgn * (M / 2 + 8), 25], [sgn * (M / 2 + 8), -25]],
                             closed=True, fc="#c8c8c8", ec=INK, lw=0.8))
    ax.add_patch(Circle((0, 0), ra, fill=False, ec=BLUE, ls="--", lw=0.7))
    ax.add_patch(Circle((0, 0), r_, fill=False, ec=RED, ls="-.", lw=0.7))
    _dim(ax, (-M / 2, 40), (M / 2, 40), f"$M_d$ = {M:.3f} mm", color=INK)
    _label(ax, f"ball (or pin) $D_M$ = {D:.2f} mm", (rM, D / 2 * 0.7), (M / 2 + 15, 75))
    _label(ax, "micrometer anvils", (M / 2 + 8, -20), (M / 2 + 15, -55))
    _label(ax, "tip circle: balls stand out past it", (ra * np.cos(0.6), ra * np.sin(0.6)), (M / 2 + 15, 105), color=BLUE)
    _label(ax, "reference circle: the ball touches the\nflanks near here", (r_ * np.cos(-0.6), r_ * np.sin(-0.6)), (M / 2 + 15, -95), color=RED)
    ax.set_title(f"Dimension over balls (z = {gm.z}, even: balls in opposite spaces)", fontsize=10)
    ax.set_xlim(-M / 2 - 20, M / 2 + 140); ax.set_ylim(-ra - 15, ra + 20); ax.set_aspect("equal"); ax.axis("off")
    plt.tight_layout()
_meas_figure()


**Span measurement (external gears).** A disc micrometer touching two flanks $k$ teeth apart lies along a base tangent, so

$$W_k = (k-1)\,p_b + s_b,\qquad s_b = d_b\left(\frac{s}{d} + \operatorname{inv}\alpha\right).$$

$k$ is chosen so that the anvils touch near the middle of the tooth depth:

$$k = \operatorname{round}\left[\frac{z}{\pi}\left(\tan\alpha_x - \frac{2x\tan\alpha}{z} - \operatorname{inv}\alpha\right) + 0.5\right],\qquad \cos\alpha_x = \frac{d_b}{d + 2xm}.$$

**Dimension over balls (external) or between balls (internal).** A ball of diameter $D_M$ in a tooth space has its centre at pressure angle $\alpha_M$ with (signed $z$, $d$, $d_b$)

$$\operatorname{inv}\alpha_M = \frac{s}{d} + \operatorname{inv}\alpha + \frac{D_M}{d_b} - \frac{\pi}{z}$$

and the measurement across two balls is

$$M_d = \frac{d_b}{\cos\alpha_M} + D_M\ \ (z \text{ even}),\qquad M_d = \frac{d_b\cos(\pi/2z)}{\cos\alpha_M} + D_M\ \ (z \text{ odd}).$$

For an internal gear the same formulas with negative $z$, $d$, $d_b$ give $M_d = d_b/\cos\alpha_M - D_M$ in positive sizes. The ball touches the flank where $\tan\alpha_c = \tan\alpha_M - D_M/d_b$. `ball_diameter` picks the ball that touches on the reference circle: with $\eta = \pi/|z| - s/d$ the half-angle of the tooth space,

$$D_M = d_b\,[\tan(\alpha + \eta) - \tan\alpha]\ \text{(external)},\qquad D_M = d_b\,[\tan\alpha - \tan(\alpha - \eta)]\ \text{(internal)},$$

rounded to 0.01 mm, a step that measuring-pin sets come in. If the balls would then sit below the tips (common with a positive shift, which raises the tip of an external gear, and on internal gears), the ball is enlarged until it stands out past the tips by $0.2\,m$, so the micrometer touches the balls and not the teeth.

In equations, $D_M$ is increased in 0.01 mm steps until

$$M_d - d_a \ge 0.2\,m\ \text{(external)},\qquad d_a - M_d \ge 0.2\,m\ \text{(internal)},$$

and the diameter at which the ball then touches the flank, reported so you can check it lies on the working involute ($d_{Ff} \le d_c \le d_a$), is

$$d_c = \frac{d_b}{\cos\alpha_c},\qquad \tan\alpha_c = \tan\alpha_M - \frac{D_M}{d_b}.$$

The span tooth count is limited to $2 \le k \le z - 1$. All three functions accept a tooth thickness $s$ other than the nominal one, so the same formulas give the upper and lower inspection limits once the thickness allowances are applied ($s + A_{sne}$, $s + A_{sni}$ in `gears.manufacturing`).

In [ ]:
#| export
def span_teeth(g: GearGeometry) -> int:
    "Number of teeth k to span for the base tangent length of an external gear."
    if g.internal:
        raise ValueError("span measurement is for external gears; use ball_measurement for internal gears")
    a = g.alpha
    ax = math.acos(min(1.0, g.db / (g.d + 2 * g.x * g.m)))
    k = round(g.z / math.pi * (math.tan(ax) - 2 * g.x * math.tan(a) / g.z - inv(a)) + 0.5)
    return int(min(max(k, 2), g.z - 1))


def span_measurement(g: GearGeometry, s: float | None = None, k: int | None = None) -> float:
    "Base tangent length W_k [mm] over k teeth for reference tooth thickness s (default nominal)."
    k = span_teeth(g) if k is None else k
    s = g.s if s is None else s
    sb = g.db * (s / g.d + inv(g.alpha))
    return (k - 1) * g.pb + sb


def ball_diameter(g: GearGeometry, protrude: float = 0.2) -> float:
    """
    Ball or pin diameter [mm], rounded to 0.01 mm: touches the flanks near the reference circle,
    enlarged if needed so the balls stand out past the tips by at least `protrude`·m
    (otherwise a micrometer anvil would rest on the tips instead of the balls).
    """
    eta = math.pi / g.z - g.s / g.d
    a = g.alpha
    if g.internal:
        D = g.db * (math.tan(a) - math.tan(a - eta))
    else:
        D = g.db * (math.tan(a + eta) - math.tan(a))
    D = round(D, 2)
    for _ in range(400):
        M, _ = ball_measurement(g, D)
        clear = (g.da - M) if g.internal else (M - g.da)
        if clear >= protrude * g.m:
            break
        D = round(D + 0.01, 2)
    return D


def ball_measurement(g: GearGeometry, D: float | None = None, s: float | None = None) -> tuple[float, float]:
    """
    Dimension over two balls (external) or between two balls (internal) [mm].

    Returns (M_d, d_contact): the measurement and the diameter where the ball touches the flank.
    """
    D = ball_diameter(g) if D is None else D
    s = g.s if s is None else s
    sg = -1.0 if g.internal else 1.0
    z, d, db = sg * g.z, sg * g.d, sg * g.db
    aM = inv_inverse(s / d + float(inv(g.alpha)) + D / db - math.pi / z)
    factor = 1.0 if g.z % 2 == 0 else math.cos(math.pi / (2 * g.z))
    M = abs(db * factor / math.cos(aM) + D)
    tan_c = math.tan(aM) - D / db
    d_contact = abs(db) / math.cos(math.atan(tan_c))
    return M, d_contact

In [ ]:
# Check against the span tables: z = 30, x = 0 → k = 4, W* = 10.7526 per unit module
g30 = GearGeometry(z=30, m=2)
assert span_teeth(g30) == 4 and abs(span_measurement(g30) / 2 - 10.7526) < 1e-4
# Ball touching at the reference circle
for g in (sun, ring, g30, GearGeometry(z=25, m=1, x=-0.2)):
    D = ball_diameter(g)
    M, dc = ball_measurement(g, D)
    print(f"{g.name:5s} z={g.z:3d}  D_M = {D:.2f}  M_d = {M:.4f}  tip d_a = {g.da:.3f}  contact at d = {dc:.3f} (ref {g.d:.3f})")
    assert (g.da - M if g.internal else M - g.da) >= 0.2 * g.m - 1e-9
    assert min(g.da, g.d_form) - 1e-9 <= dc <= max(g.da, g.d_form) + 1e-9      # contact on the working flank

A thinner tooth (the tolerance allowances of `gears.manufacturing`) must give a smaller span and a smaller dimension over balls on an external gear, and a *larger* dimension between balls on an internal gear:

In [ ]:
ds = -0.05
assert span_measurement(sun, sun.s + ds) < span_measurement(sun)
assert ball_measurement(sun, 2.8, sun.s + ds)[0] < ball_measurement(sun, 2.8)[0]
assert ball_measurement(ring, 2.5, ring.s + ds)[0] > ball_measurement(ring, 2.5)[0]
# span change = Δs·cos α exactly
assert math.isclose(span_measurement(sun) - span_measurement(sun, sun.s + ds), -ds * math.cos(sun.alpha))

## Meshing Pair: Theory

Two gears mesh along the **line of action**, the common tangent of their base circles ($T_1T_2$). Contact starts where the driven gear's tip circle cuts it (A) and ends where the driver's tip circle does (E); AE is the path of contact. The line of action is inclined at the operating pressure angle $\alpha_w$ to the tangent at the pitch point C, and the axes are $a_w$ apart.

In [ ]:
#| echo: false
def _mesh_figure():
    fig, ax = plt.subplots(figsize=(12, 6.2))
    g1, g2 = _pkg.GearGeometry(z=14, m=10, x=0.3, name="pinion"), _pkg.GearGeometry(z=22, m=10, x=-0.1, name="wheel")
    mc = _pkg.mesh_check(g1, g2)
    aw, alw = mc.a_w, mc.alpha_w
    c1, c2 = np.array([0.0, 0.0]), np.array([0.0, aw])
    ds = -0.8                                     # visible backlash
    P1 = _rot(gear_outline(g1, ds, n_flank=50), np.pi / 2)
    # wheel: a tooth space faces the pinion tooth at the pitch point
    P2 = _rot(gear_outline(g2, ds, n_flank=50), -np.pi / 2 + np.pi / g2.z) + c2
    ax.add_patch(Polygon(P1, closed=True, fc="#d8e8d0", ec=INK, lw=0.8))
    ax.add_patch(Polygon(P2, closed=True, fc=C_GEAR, ec=INK, lw=0.8))
    rw1 = aw * g1.z / (g1.z + g2.z)
    C = np.array([0, rw1])
    for c, gg in ((c1, g1), (c2, g2)):
        _circ(ax, gg.db / 2, "--", GREEN, 0.8, c)
        _circ(ax, gg.da / 2, ":", BLUE, 0.9, c)
    _circ(ax, rw1, "-.", RED, 0.8, c1)
    _circ(ax, aw - rw1, "-.", RED, 0.8, c2)
    T1 = c1 + g1.db / 2 * np.array([np.sin(alw), np.cos(alw)])
    T2 = c2 - g2.db / 2 * np.array([np.sin(alw), np.cos(alw)])
    ax.plot([T1[0], T2[0]], [T1[1], T2[1]], color=GREEN, lw=1.3)
    for T, nm, cc in ((T1, "$T_1$", c1), (T2, "$T_2$", c2)):
        ax.plot(*T, "o", color=GREEN, ms=3.5); ax.text(T[0] + 2.5, T[1] + (-7 if T is T1 else 1), nm, color=GREEN, fontsize=10)
    def hit(c, ra):
        u = (T2 - T1) / np.linalg.norm(T2 - T1)
        f = T1 - c
        bq = 2 * f @ u; cq = f @ f - ra ** 2
        ts = [(-bq + s_ * math.sqrt(bq * bq - 4 * cq)) / 2 for s_ in (1, -1)]
        return min((T1 + t_ * u for t_ in ts), key=lambda p_: np.linalg.norm(p_ - C))
    A, E = hit(c2, g2.da / 2), hit(c1, g1.da / 2)
    ax.plot([A[0], E[0]], [A[1], E[1]], color=RED, lw=4, alpha=0.55, solid_capstyle="butt")
    for p_, nm, o in ((A, "A", (-7, -6)), (E, "E", (2, 3)), (C, "C", (-3, 4))):
        ax.plot(*p_, "o", color=RED, ms=4); ax.text(p_[0] + o[0], p_[1] + o[1], nm, color=RED, fontsize=11)
    ax.plot([-60, 75], [rw1, rw1], color=INK, lw=0.5, ls="--")
    ax.add_patch(Arc(C, 84, 84, theta1=-np.degrees(alw), theta2=0, color=GREEN, lw=1.3))
    ax.text(C[0] + 44, C[1] - 9, "$\\alpha_w$", color=GREEN, fontsize=13)
    ax.plot([0, 0], [rw1 - 75, rw1 + 75], color=RED, lw=0.4, ls="-.")
    # leaders: right side, top to bottom
    xr = 110
    _label(ax, "line of action (common tangent\nof the two base circles)", T2 + 0.2 * (T1 - T2), (xr, rw1 + 48), color=GREEN)
    _label(ax, "operating pressure angle $\\alpha_w$", C + [41.5, -6], (xr, rw1 + 30), color=GREEN)
    _label(ax, "path of contact $g_\\alpha$ = AE;\ncontact ratio $\\varepsilon_\\alpha = g_\\alpha / p_b$", (A + E) / 2 + (A - E) * 0.25, (xr, rw1 + 8), color=RED)
    _label(ax, "tangent to the pitch circles at C", (60, rw1), (xr, rw1 - 14), color=INK)
    _label(ax, "line of centres (wheel centre $O_2$ above,\npinion centre $O_1$ below)", (0, rw1 - 60), (xr, rw1 - 40), color=RED)
    # leaders: left side, top to bottom
    xl = -95
    _label(ax, "wheel (driven)", c2 + (g2.d / 2 - 12) * np.array([np.cos(-2.0), np.sin(-2.0)]), (xl, rw1 + 64), ha="right")
    _label(ax, "tip circles", c2 + (g2.da / 2) * np.array([np.cos(-2.15), np.sin(-2.15)]), (xl, rw1 + 50), ha="right", color=BLUE)
    _label(ax, "pitch point C: the pitch circles\nroll on each other here", C + [-1, 1], (xl, rw1 + 30), ha="right", color=RED)
    _label(ax, "backlash $j$: play between the\nnon-driving flanks", C + [-9.5, 2.5], (xl, rw1 + 10), ha="right")
    _label(ax, "pitch circles (radii $r_{w1}, r_{w2}$)", (rw1 * np.cos(2.0), rw1 * np.sin(2.0)), (xl, rw1 - 22), ha="right", color=RED)
    _label(ax, "base circles", (g1.db / 2 * np.cos(2.2), g1.db / 2 * np.sin(2.2)), (xl, rw1 - 38), ha="right", color=GREEN)
    _label(ax, "pinion (driver)", ((g1.d / 2 - 14) * np.cos(2.0), (g1.d / 2 - 14) * np.sin(2.0)), (xl, rw1 - 56), ha="right")
    ax.text(0.5, -0.01, f"Pinion z = {g1.z}, x = {g1.x:+.1f}; wheel z = {g2.z}, x = {g2.x:+.1f}; m = {g1.m:g} mm (drawn large). "
            f"Centre distance $a_w$ = {aw:.2f} mm (reference a = {(g1.z + g2.z) * g1.m / 2:.0f} mm), "
            f"$\\alpha_w$ = {math.degrees(alw):.2f}°, $\\varepsilon_\\alpha$ = {mc.eps_alpha:.2f}", ha="center", va="top", fontsize=9, transform=ax.transAxes)
    ax.set_xlim(-245, 260); ax.set_ylim(rw1 - 80, rw1 + 75); ax.set_aspect("equal"); ax.axis("off")
    plt.tight_layout()
_mesh_figure()

Two gears with profile shifts $x_1, x_2$ mesh without backlash at the operating pressure angle (signed $z$ for an internal gear)

$$\operatorname{inv}\alpha_w = \operatorname{inv}\alpha + 2\tan\alpha\,\frac{x_1 + x_2}{z_1 + z_2}$$

and the operating centre distance is

$$a_w = \frac{(z_1 + z_2)\,m}{2}\,\frac{\cos\alpha}{\cos\alpha_w}.$$

Turned around, a required centre distance fixes the shift sum: $\cos\alpha_w = a\cos\alpha / a_w$ with $a = (z_1+z_2)m/2$, then $x_1 + x_2 = (z_1+z_2)(\operatorname{inv}\alpha_w - \operatorname{inv}\alpha)/(2\tan\alpha)$. This is how the planet–ring shift is found once the sun–planet mesh has fixed $a_w$.

**Transverse contact ratio** (gear 1 external, gear 2 external or internal):

$$\varepsilon_\alpha = \frac{\sqrt{r_{a1}^2 - r_{b1}^2} \pm \sqrt{r_{a2}^2 - r_{b2}^2} \mp a_w\sin\alpha_w}{p_b},$$

upper signs for an external pair, lower signs for an internal pair. Keep $\varepsilon_\alpha \ge 1.2$; values from 1.4 to 1.8 are usual for spur gears.

**Start of active profile and interference.** Measure lengths along the line of action. $T_1T_2 = a_w\sin\alpha_w$ is the distance between the two tangent points, and $t_i = \sqrt{r_{ai}^2 - r_{bi}^2}$ is the distance from $T_i$ to where the tip circle of gear $i$ crosses the line. Contact on gear 1 starts where the tip of gear 2 reaches it, at a distance $g_1$ from $T_1$, and likewise for gear 2:

| | External pair | Internal pair (gear 2 = ring) |
|---|---|---|
| on gear 1 | $g_1 = T_1T_2 - t_2$ | $g_1 = t_2 - T_1T_2$ |
| on gear 2 | $g_2 = T_1T_2 - t_1$ | $g_2 = T_1T_2 + t_1$ (outermost contact on the ring) |

$$d_{Nf\,i} = 2\sqrt{r_{bi}^2 + g_i^2}.$$

The mesh has **interference** if $g_1 < 0$ (and, for an external pair, $g_2 < 0$): the mating tip would have to reach past the tangent point. It also has interference if contact runs into a fillet, that is $d_{Nf1} < d_{Ff1}$ on gear 1, and on gear 2 $d_{Nf2} < d_{Ff2}$ (external) or $d_{Nf2} > d_{Ff2}$ (internal, where the fillet is outside the working flank). For the ring, $g_1 \ge 0$ is the condition that its tip lies beyond $T_1$; if not, the ring tip digs into the planet root.

**Tip clearance** is the radial gap between each tip circle and the mating root circle:

$$c_{12} = a_w - r_{a1} - r_{f2},\quad c_{21} = a_w - r_{a2} - r_{f1}\ \text{(external)};\qquad c_{12} = r_{f2} - a_w - r_{a1},\quad c_{21} = r_{a2} - a_w - r_{f1}\ \text{(internal)}.$$

It is $0.25\,m$ with the standard rack and no shift; aim for at least $0.1\,m$, since below that oil is trapped and the tip may touch the fillet.

**Pass criteria used by `MeshCheck.ok`:** $\varepsilon_\alpha \ge 1.2$, no interference, and both clearances above zero. Both gears of a pair must have the same module and pressure angle.

In [ ]:
#| export
def operating_pressure_angle(g1: GearGeometry, g2: GearGeometry) -> float:
    "Operating pressure angle α_w [rad] of a backlash-free mesh (g2 may be internal)."
    if g1.internal:
        raise ValueError("g1 must be the external gear of the pair")
    _check_pair(g1, g2)
    zsum = g1.zs + g2.zs
    return inv_inverse(float(inv(g1.alpha)) + 2 * math.tan(g1.alpha) * (g1.x + g2.x) / zsum)


def centre_distance(g1: GearGeometry, g2: GearGeometry) -> float:
    "Operating centre distance a_w [mm] of a backlash-free mesh."
    aw = operating_pressure_angle(g1, g2)
    return abs((g1.zs + g2.zs) * g1.m / 2 * math.cos(g1.alpha) / math.cos(aw))


def shift_sum_for_centre_distance(z1: int, z2_signed: int, m: float, a_w: float,
                                  alpha: float = math.radians(20.0)) -> float:
    "Profile shift sum x1 + x2 that gives centre distance a_w [mm] (z2 negative for an internal gear)."
    zsum = z1 + z2_signed
    a = abs(zsum) * m / 2
    c = a * math.cos(alpha) / a_w
    if not 0 < c < 1:
        raise ValueError("centre distance not reachable with an involute mesh")
    aw = math.acos(c)
    return zsum * (float(inv(aw)) - float(inv(alpha))) / (2 * math.tan(alpha))


def contact_ratio(g1: GearGeometry, g2: GearGeometry) -> float:
    "Transverse contact ratio ε_α of the pair (g1 external, g2 external or internal)."
    aw = operating_pressure_angle(g1, g2)
    a_w = centre_distance(g1, g2)
    t1 = math.sqrt(max(0.0, (g1.da / 2) ** 2 - (g1.db / 2) ** 2))
    t2 = math.sqrt(max(0.0, (g2.da / 2) ** 2 - (g2.db / 2) ** 2))
    if g2.internal:
        return (t1 - t2 + a_w * math.sin(aw)) / g1.pb
    return (t1 + t2 - a_w * math.sin(aw)) / g1.pb


@dataclass(frozen=True)
class MeshCheck:
    """
    Geometry checks of one mesh. Lengths in mm.

    alpha_w        : operating pressure angle [rad]
    a_w            : operating centre distance
    eps_alpha      : transverse contact ratio
    d_Nf1, d_Nf2   : start of active profile on each gear (g2: for an internal gear, the outermost contact diameter)
    d_Ff1, d_Ff2   : root form diameters
    clearance12    : g1 tip to g2 root
    clearance21    : g2 tip to g1 root
    interference   : True if contact runs into a fillet or (internal) the ring tip lies inside T1
    """
    alpha_w: float
    a_w: float
    eps_alpha: float
    d_Nf1: float
    d_Nf2: float
    d_Ff1: float
    d_Ff2: float
    clearance12: float
    clearance21: float
    interference: bool

    @property
    def ok(self) -> bool:
        m_ok = self.eps_alpha >= 1.2 and not self.interference
        return m_ok and min(self.clearance12, self.clearance21) > 0


def mesh_check(g1: GearGeometry, g2: GearGeometry) -> MeshCheck:
    "Contact ratio, active profile, interference and tip clearance of a mesh."
    aw = operating_pressure_angle(g1, g2)
    a_w = centre_distance(g1, g2)
    rb1, rb2 = g1.db / 2, g2.db / 2
    ra1, ra2 = g1.da / 2, g2.da / 2
    if ra2 < rb2 - 1e-9 and g2.internal:
        raise ValueError(f"{g2.name}: tip circle inside the base circle; raise k_tip")
    t1 = math.sqrt(max(0.0, ra1 ** 2 - rb1 ** 2))
    t2 = math.sqrt(max(0.0, ra2 ** 2 - rb2 ** 2))
    T = a_w * math.sin(aw)                        # T1T2
    if g2.internal:
        g_1 = t2 - T                              # ring tip, measured from T1
        g_2 = T + t1                              # planet tip, measured from T2
        d_Nf1 = 2 * math.hypot(rb1, g_1)
        d_Nf2 = 2 * math.hypot(rb2, g_2)
        interf = g_1 < 0 or d_Nf1 < g1.d_form - 1e-6 or d_Nf2 > g2.d_form + 1e-6
        c12 = g2.df / 2 - a_w - ra1
        c21 = ra2 - a_w - g1.df / 2
    else:
        g_1 = T - t2
        g_2 = T - t1
        d_Nf1 = 2 * math.hypot(rb1, max(g_1, 0.0))
        d_Nf2 = 2 * math.hypot(rb2, max(g_2, 0.0))
        interf = g_1 < 0 or g_2 < 0 or d_Nf1 < g1.d_form - 1e-6 or d_Nf2 < g2.d_form - 1e-6
        c12 = a_w - ra1 - g2.df / 2
        c21 = a_w - ra2 - g1.df / 2
    return MeshCheck(aw, a_w, contact_ratio(g1, g2), d_Nf1, d_Nf2, g1.d_form, g2.d_form, c12, c21, interf)


def _check_pair(g1: GearGeometry, g2: GearGeometry):
    if not math.isclose(g1.m, g2.m) or not math.isclose(g1.alpha, g2.alpha):
        raise ValueError("gears of a pair need the same module and pressure angle")

### Example

The sun above with a 25-tooth planet at $x = -0.3$ (a zero-sum pair, so $a_w = a$), and the ring meshing with the planet at the same centre distance. The ring then needs $x = +0.3$, and its lengthened teeth reach into the planet fillet; shortening them by $0.2\,m$ (`k_tip`) clears it at the cost of some contact ratio.

In [ ]:
planet = GearGeometry(z=25, m=1.5, x=-0.3, b=20, name="planet")
sp = mesh_check(sun, planet)
xr = shift_sum_for_centre_distance(planet.z, -ring.z, 1.5, sp.a_w) - planet.x
pr0 = mesh_check(planet, ring.with_(x=xr))
print(f"ring at x = {xr:+.2f}, no tip reduction: interference = {pr0.interference} "
      f"(contact starts at d = {pr0.d_Nf1:.3f} on the planet, its form circle is {pr0.d_Ff1:.3f})")
ring2 = ring.with_(x=xr, k_tip=0.2)      # shorten the ring teeth by 0.2 m
pr = mesh_check(planet, ring2)
print(f"sun–planet:  a_w = {sp.a_w:.4f}  α_w = {math.degrees(sp.alpha_w):.3f}°  ε_α = {sp.eps_alpha:.3f}  "
      f"clearances {sp.clearance12:.3f}/{sp.clearance21:.3f} mm  ok={sp.ok}")
print(f"planet–ring: a_w = {pr.a_w:.4f}  α_w = {math.degrees(pr.alpha_w):.3f}°  ε_α = {pr.eps_alpha:.3f}  "
      f"clearances {pr.clearance12:.3f}/{pr.clearance21:.3f} mm  x_ring = {xr:+.3f}  ok={pr.ok}")
assert math.isclose(sp.a_w, (17 + 25) * 1.5 / 2) and math.isclose(pr.a_w, sp.a_w)
assert math.isclose(xr, 0.3) and pr0.interference and sp.ok and pr.ok

---

## Tests

In [ ]:
#| hide
# Standard external pair, x = 0: α_w = α and a_w = a
g1, g2 = GearGeometry(z=20, m=2), GearGeometry(z=40, m=2)
assert math.isclose(operating_pressure_angle(g1, g2), math.radians(20)) and math.isclose(centre_distance(g1, g2), 60)
# Contact ratio of 20/40 standard pair (textbook ≈ 1.64)
assert abs(contact_ratio(g1, g2) - 1.64) < 0.01
# Positive shift sum increases centre distance and α_w; shift_sum_for_centre_distance inverts it
g1s, g2s = g1.with_(x=0.3), g2.with_(x=0.2)
aw = centre_distance(g1s, g2s)
assert aw > 60 and math.isclose(shift_sum_for_centre_distance(20, 40, 2, aw), 0.5, abs_tol=1e-9)
# Internal: x = 0 on both gives a = (zr − zp) m / 2
gr = GearGeometry(z=60, m=2, internal=True)
assert math.isclose(centre_distance(g1, gr), 40.0)
# Internal contact ratio is higher than external for the same pinion
assert contact_ratio(g1, gr) > contact_ratio(g1, g2)
# Thickness on reference circle via s_at equals s
for g in (sun, ring2, planet):
    assert math.isclose(g.s_at(g.d), g.s, rel_tol=1e-12)
# Odd tooth count over-ball formula reduces to even formula for large z
gE, gO = GearGeometry(z=200, m=1), GearGeometry(z=201, m=1)
assert abs(ball_measurement(gO, 1.7)[0] - ball_measurement(gE, 1.7)[0] - 1.0) < 0.02
# A mesh with too little clearance is flagged
bad = mesh_check(GearGeometry(z=20, m=2, k_tip=-0.3), GearGeometry(z=40, m=2, k_tip=-0.3))
assert not bad.ok

## References

- ISO 21771:2007 — *Gears — Cylindrical involute gears and gear pairs — Concepts and geometry*. Signed tooth counts for internal gears, profile shift, operating pressure angle.
- ISO 53:1998 — *Standard basic rack tooth profile* (profile A: α = 20°, h_aP = 1.0 m, h_fP = 1.25 m, ρ_fP = 0.38 m).
- ISO 54:1996 — *Modules*.
- DIN 3960:1987 — *Definitions, parameters and equations for involute cylindrical gears*: span and over-ball measurements.
- Linke, H. (2010). *Stirnradverzahnung* (2nd ed.). Hanser — interference checks for internal gear pairs.